## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

#include <iostream>
#include <vector>
#include <algorithm>
#include <cassert>

using namespace std;

class MagicSolver {
    int n;
    int a, b;
    int l;
    vector<int> board;
    vector<int> pos;
    vector<pair<int, int>> ans;

    void refresh_pos() {
        for (int i = 0; i < n; ++i) pos[board[i]] = i;
    }

    void do_swap_magic() {
        ans.emplace_back(0, 0);
        int pa = pos[a], pb = pos[b];
        swap(board[pa], board[pb]);
        pos[a] = pb;
        pos[b] = pa;
    }

    void do_add(int x) {
        if (x == 0) return;
        ans.emplace_back(2, x);
        for (int i = 0; i < n; ++i) board[i] = (board[i] + x) % n;
        refresh_pos();
    }

    void do_xor(int x) {
        if (x == 0) return;
        ans.emplace_back(1, x);
        for (int i = 0; i < n; ++i) board[i] ^= x;
        refresh_pos();
    }

    void calc(int u, int v, int &pu, int &pv) const {
        int delta = (v - u + n - l + n) % n;
        pu = pv = 0;
        for (int step = n / 2; step >= 2 * l; step /= 2) {
            if (delta >= step) {
                delta -= step;
                pv += step / 2;
            } else {
                pu += step / 2;
            }
        }
        pu += n / 2;
        pu += (u & (l - 1));
        pv += (u & (l - 1));
    }

    void swap_numbers(int c, int d) {
        if (c == d) return;
        if ((c / l) % 2 == (d / l) % 2) {
            int pivot;
            if ((c / l) % 2 == 0)
                pivot = (c & (l - 1)) + l;
            else
                pivot = (c & (l - 1));
            swap_numbers(c, pivot);
            swap_numbers(d, pivot);
            swap_numbers(c, pivot);
            return;
        }

        int pa, pb, pc, pd;
        calc(a, b, pa, pb);
        calc(c, d, pc, pd);

        do_add((pc - c + n) % n);
        do_xor(pc ^ pa);
        do_add((a - pa + n) % n);
        do_swap_magic();
        do_add((pa - a + n) % n);
        do_xor(pc ^ pa);
        do_add((c - pc + n) % n);
    }

    struct PermResult {
        bool valid;
        vector<int> ops;
    };

    PermResult solve_lowbits(const vector<int>& arr, int m) {
        vector<bool> seen(m, false);
        for (int i = 0; i < m; ++i) {
            int x = arr[i];
            if (x < 0 || x >= m || seen[x]) return {false, {}};
            seen[x] = true;
        }
        if (m == 1) return {true, {}};

        vector<int> even(m/2), odd(m/2);
        for (int i = 0; i < m/2; ++i) {
            even[i] = arr[2*i] / 2;
            odd[i]  = arr[2*i+1] / 2;
        }

        auto res_even = solve_lowbits(even, m/2);
        auto res_odd  = solve_lowbits(odd,  m/2);
        if (!res_even.valid || !res_odd.valid) return {false, {}};

        vector<int> ops;
        if (arr[0] % 2) ops.push_back(m == 2 ? 1 : -1);

        int tb = 0;
        for (int op : res_even.ops) {
            if (op > 0) {
                ops.push_back(-1);
                ops.push_back(1);
            } else {
                ops.push_back(op * 2);
                tb ^= -op * 2;
            }
        }
        if (tb) ops.push_back(-tb);

        int tc = 0;
        for (int op : res_odd.ops) {
            if (op > 0) {
                ops.push_back(1);
                ops.push_back(-1);
            } else {
                ops.push_back(op * 2);
                tc ^= -op * 2;
            }
        }
        if ((tc & (m/2)) != (tb & (m/2))) {
            for (int i = 0; i < m/4; ++i) {
                ops.push_back(-1);
                ops.push_back(1);
            }
        }
        if (tb >= m/2) tb -= m/2;
        if (tc >= m/2) tc -= m/2;
        if (tb != tc) return {false, {}};

        vector<int> merged;
        for (int op : ops) {
            if (merged.empty()) merged.push_back(op);
            else if (op < 0 && merged.back() < 0) {
                int new_xor = -((-merged.back()) ^ (-op));
                merged.back() = new_xor;
                if (merged.back() == 0) merged.pop_back();
            } else {
                merged.push_back(op);
            }
        }
        return {true, merged};
    }

public:
    MagicSolver(int n_, int a_, int b_, const vector<int>& init)
        : n(n_), a(a_), b(b_), board(init), pos(n_) {
        refresh_pos();
        l = (a - b + n) % n;
        l = l & -l;
        if (l == 0) l = n;
    }

    bool solve() {
        if (l > 1) {
            vector<int> low(l);
            for (int i = 0; i < l; ++i) low[i] = board[i] & (l - 1);
            auto res = solve_lowbits(low, l);
            if (!res.valid) return false;
            for (int op : res.ops) {
                if (op > 0) do_add(op);
                else        do_xor(-op);
            }
        }

        for (int rem = 0; rem < l; ++rem) {
            vector<int> expected, actual;
            for (int j = rem; j < n; j += l) {
                expected.push_back(j);
                actual.push_back(board[j]);
            }
            sort(actual.begin(), actual.end());
            if (actual != expected) return false;
        }

        for (int rem = 0; rem < l; ++rem) {
            for (int j = rem; j < n; j += l) {
                while (board[j] != j) {
                    swap_numbers(j, board[j]);
                }
            }
        }

        for (int i = 0; i < n; ++i)
            if (board[i] != i) return false;
        return true;
    }

    void print_answer() const {
        cout << ans.size() << '\n';
        for (auto [type, param] : ans) {
            if (type == 0)      cout << "0\n";
            else if (type == 1) cout << "1 " << param << '\n';
            else                cout << "2 " << param << '\n';
        }
    }
};

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n, a, b;
    cin >> n >> a >> b;
    vector<int> init(n);
    for (int i = 0; i < n; ++i) cin >> init[i];

    MagicSolver solver(n, a, b, init);
    if (!solver.solve()) {
        cout << "-1\n";
        return 0;
    }
    solver.print_answer();
    return 0;
}

## B 长跑

#include <iostream>
#include <vector>
#include <algorithm>
#include <climits>

using namespace std;

struct station {
    int p;
    int c;
};

const int INF = INT_MAX / 2;

bool cmp(station x, station y) {
    if (x.p != y.p) {
        return x.p < y.p;
    }
    return x.c < y.c;
}

int main() {
    int N, L, Maxn, S;
    while (cin >> N >> L >> Maxn >> S) {
        vector<station> a;
        
        for (int i = 0; i < N; ++i) {
            int p, c;
            cin >> p >> c;
            if (p <= L) {
                a.push_back({p, c});
            }
        }
        
        N = a.size();
        sort(a.begin(), a.end(), cmp);

        vector<int> dp(N, INF);  // dp[i]：到达第i个补给站并补满体力的最小花费

        if (N > 0 && a[0].p <= Maxn) {
            dp[0] = a[0].c;
        }

        for (int i = 1; i < N; i ++) {
            for (int j = i - 1; j >= 0; j --) {
                if (a[i].p - a[j].p > Maxn) {
                    break;
                }
    
                if (dp[j] != INF) {
                    dp[i] = min(dp[i], dp[j] + a[i].c);
                }
            }
            
            if (a[i].p <= Maxn) {
                dp[i] = min(dp[i], a[i].c);
            }
        }

        bool ok = false;
        
        if (L <= Maxn) {
            ok = true;
        }
        
        for (int i = 0; i < N && !ok; ++i) {
            if (dp[i] != INF && (L - a[i].p) <= Maxn && dp[i] <= S) {
                ok = true;
            }
        }

        cout << (ok ? "Yes" : "No") << endl;
    }

    return 0;
}

## C 最长回文

In [ ]:
#include <iostream>
#include <vector>

using namespace std;

vector<int> p1, p2;

void manacher(string &t, vector<int> &p) {
    string s = "$";
    for (char c : t) {
        s += '#';
        s += c;
    }
    s += '#';

    int len = (int)s.size();
    p.assign(len + 1, 0);

    int mx = 0, id = 0;
    for (int i = 1; i < len; i++) {
        p[i] = (i < mx ? min(mx - i, p[2 * id - i]) : 1);
        while (i - p[i] >= 1 && i + p[i] < len && s[i - p[i]] == s[i + p[i]]) {
            p[i]++;
        }
        if (i + p[i] > mx) {
            mx = i + p[i];
            id = i;
        }
    }
    t = s;
}

int main() {
    ios::sync_with_stdio(false);
    cin.tie(nullptr);

    int n;
    string A, B;
    cin >> n >> A >> B;

    manacher(A, p1);
    manacher(B, p2);

    int ans = 0;

    for (int i = 1; i < 2 * n + 2; i++) {
        int temp = 0;
        if (i - 2 >= 0) temp = max(p1[i], p2[i - 2]);
        else temp = p1[i];

        while (i - temp >= 0 &&
               i - 2 + temp >= 0 &&
               i - temp < (int)A.size() &&
               i - 2 + temp < (int)B.size() &&
               A[i - temp] == B[i - 2 + temp]) {
            temp++;
        }
        ans = max(ans, temp);
    }

    cout << ans - 1 << '\n';
    return 0;
}

## D 优惠券

#include <iostream>
#include <string.h>
#include <vector>
#include <set>
#include <algorithm>
#include <cstdio>
#include <map>

using namespace std;

char c[555555];
int id[555555];

int main() {
    int n;
    while (~scanf("%d", &n)) {
        int ans = -1;
        set<int> pos;
        int flag = 1;
        map<int, int> a;
        char g[2];
        for (int i = 1; i <= n; i++) {
            scanf("%s", g);
            c[i] = g[0];
            if (c[i] != '?') {
                scanf("%d", &id[i]);
            }
            if (flag == 0) {
                continue;
            }
            if (c[i] == '?') {
                pos.insert(i);
                continue;
            }

            if (a[id[i]] != 0) {
                if (c[a[id[i]]] == c[i]) {
                    set<int>::iterator it;
                    it = pos.lower_bound(a[id[i]]);
                    if (it == pos.end()) {
                        ans = i;
                        flag = 0;
                    } else {
                        pos.erase(it);
                    }
                }
                a[id[i]] = i;
            } else {
                if (c[i] == 'O') {
                    set<int>::iterator it;
                    it = pos.lower_bound(0);
                    if (it == pos.end()) {
                        ans = i;
                        flag = 0;
                    } else {
                        pos.erase(it);
                    }
                }
                a[id[i]] = i;
            }
        }
        printf("%d\n", ans);
    }
    return 0;
}

## E 任意点

#include <iostream>
#include <vector>
#include <unordered_map>
using namespace std;

vector<int> parent;

int find(int x) {
    if (parent[x] != x) {
        parent[x] = find(parent[x]);
    }
    return parent[x];
}

void unite(int x, int y) {
    parent[find(x)] = find(y);
}

int main() {
    int n;
    cin >> n;
    if (n == 0) {
        cout << 0 << endl;
        return 0;
    }

    vector<pair<int, int>> points(n);
    for (int i = 0; i < n; ++i) {
        cin >> points[i].first >> points[i].second;
    }

    parent.resize(n);
    for (int i = 0; i < n; ++i) {
        parent[i] = i;
    }

    unordered_map<int, int> x_first;
    for (int i = 0; i < n; ++i) {
        int x = points[i].first;
        if (x_first.count(x)) {
            unite(i, x_first[x]);
        } else {
            x_first[x] = i;
        }
    }

    unordered_map<int, int> y_first;
    for (int i = 0; i < n; ++i) {
        int y = points[i].second;
        if (y_first.count(y)) {
            unite(i, y_first[y]);
        } else {
            y_first[y] = i;
        }
    }

    unordered_map<int, bool> root_cnt;
    for (int i = 0; i < n; ++i) {
        root_cnt[find(i)] = true;
    }
    int components = root_cnt.size();

    cout << components - 1 << endl;

    return 0;
}

## F 通配符匹配

#include<iostream>
#include<cstring>
#include<string>

#define ull unsigned long long
#define MIKU 0

using namespace std;
 
const ull P = 131;
 
int n, cnt;
char sign[15];
string s[15];
ull v[15], h[100005], p[100005];
bool dp[15][100005], tag[15][100005];
 
ull get_hash(string s) {
	ull res = 0;
	int len = s.size();
	for(int i=0; i<len; i++) res = res * P + s[i];
	return res;
}
 
void pre(string s) {
	int len = s.size();
	h[1] = s[0];
	for(int i=2; i<=len; i++) h[i] = h[i-1] * P + s[i-1];  
}
 
bool check(string str) {
	dp[0][0] = 1; 
	int len = str.size();
	for(int j=0; j<len; j++) {
		for(int i=0; i<=cnt; i++) {
			if(!dp[i][j]) continue;
			ull v2 = h[j+s[i+1].size()] - h[j] * p[s[i+1].size()];
			if(v2 == v[i+1]) {
				if(sign[i+1] == '?') dp[i+1][j+s[i+1].size()+1] = 1;
				else dp[i+1][j+s[i+1].size()] = 1, tag[i+1][j+s[i+1].size()] = 1;
			}
			if(tag[i][j]) dp[i][j+1] = 1, tag[i][j+1] = 1;
		}
	}
	return dp[cnt][len];
}
 
int main() {
	char ch = getchar();
	string str = "";
	while(ch != '\n') {
		if(ch == '*' || ch == '?') sign[++cnt] = ch, ch = getchar(), s[cnt] = str, str = "";
		else str += ch, ch = getchar();
	}
	sign[++cnt] = '?'; 
	if(str.size()) s[cnt] = str;
	cin >> n;
	for(int i=1; i<=cnt; i++) v[i] = get_hash(s[i]);
	p[0] = 1;
	for(int i=1; i<=100000; i++) p[i] = p[i-1] * P;
	for(int i=1; i<=n; i++) {
		memset(h, 0, sizeof(h));
		memset(dp, 0, sizeof(dp)), memset(tag, 0, sizeof(tag));
		cin>>str;
		str += '%';  
		pre(str);
		if(check(str)) cout << "YES\n";
		else cout << "NO\n";
	} 
	return MIKU;
}

## G 汉诺塔

#include <iostream>
#include <vector>
#include <string>
#include <algorithm>

using namespace std;

pair<int, int> parse_op(const string& op) {
    int from = op[0] - 'A';
    int to = op[1] - 'A';
    return {from, to};
}

long long simulate(int n, int target, const vector<pair<int, int>>& ops) {
    vector<vector<int>> piles(3);
    for (int i = n; i >= 1; i--) {
        piles[0].push_back(i);
    }
    
    int last_disk = -1;
    long long steps = 0;
    
    while (true) {
        bool done = true;
        if (!piles[0].empty()) done = false;
        if (piles[target].size() < (size_t)n) done = false;
        if (done) break;
        
        bool moved = false;
        for (const auto& op : ops) {
            int from = op.first;
            int to = op.second;
            
            if (piles[from].empty()) continue;
            int disk = piles[from].back();
            if (disk == last_disk) continue;
            if (!piles[to].empty() && piles[to].back() < disk) continue;
            
            piles[from].pop_back();
            piles[to].push_back(disk);
            last_disk = disk;
            steps++;
            moved = true;
            break;
        }
        
        if (!moved) break;
    }
    
    return steps;
}

int main() {
    int n;
    cin >> n;
    
    vector<string> op_str(6);
    for (int i = 0; i < 6; i++) {
        cin >> op_str[i];
    }
    
    vector<pair<int, int>> ops(6);
    for (int i = 0; i < 6; i++) {
        ops[i] = parse_op(op_str[i]);
    }
    
    int idx_ab = -1, idx_ac = -1;
    for (int i = 0; i < 6; i++) {
        if (op_str[i] == "AB") idx_ab = i;
        if (op_str[i] == "AC") idx_ac = i;
    }
    int target = (idx_ab < idx_ac) ? 1 : 2;
    
    if (n == 1) {
        cout << 1 << endl;
        return 0;
    }
    
    long long f2 = simulate(2, target, ops);
    long long f3 = simulate(3, target, ops);
    
    long long k = (f3 - f2) / (f2 - 1);
    long long b = f2 - k;
    
    vector<long long> f(n + 1);
    f[1] = 1;
    if (n >= 2) f[2] = f2;
    for (int i = 3; i <= n; i++) {
        f[i] = k * f[i-1] + b;
    }
    
    cout << f[n] << endl;
    
    return 0;
}

## H 马步距离

#include<stdio.h>
#include<math.h>
#include<map>
#include<queue>
#include<algorithm>

using namespace std;
map<pair<int, int>, int> p;
typedef struct
{
	int x, y;
	int sum;
}Point;
Point now, temp, c;
queue<Point> q;
int dir[8][2] = {1,2,1,-2,-1,2,-1,-2,2,1,2,-1,-2,1,-2,-1};
int main(void)
{
	int i, ex, ey, sx, sy, sum;
	scanf("%d%d%d%d", &sx, &sy, &ex, &ey);
	sx = abs(ex-sx);
	sy = abs(ey-sy);
	sum = 0;
	while(sx>=15 || sy>=15)
	{
		if(sx>sy)
			sx -= 2, sy = abs(sy-1);
		else
			sy -= 2, sx = abs(sx-1);
		sum++;
	}
	now.x = sx, now.y = sy, now.sum = 0;
	q.push(now);
	while(q.empty()==0)
	{
		now = q.front();
		q.pop();
		if(now.x==0 && now.y==0)
			break;
		temp.sum = now.sum+1;
		for(i=0;i<=7;i++)
		{
			temp.x = now.x+dir[i][0];
			temp.y = now.y+dir[i][1];
			if(p.count(make_pair(temp.x, temp.y))==0)
			{
				p[make_pair(temp.x, temp.y)] = 1;
				q.push(temp);
			}
		}
	}
	printf("%d\n", sum+now.sum);
	return 0;
}

## I 直方图最大矩形

class Solution {
public:
    /**
     * 代码中的类名、方法名、参数名已经指定，请勿修改，直接返回方法规定的值即可
     *
     * 
     * @param heights int整型vector 
     * @return int整型
     */
    int largestRectangleArea(vector<int>& heights) {
        heights.insert(heights.begin(), 0);
        heights.push_back(0);
        int n = heights.size();
        stack<int> stk;
        int maxArea = 0;

        for (int i = 0; i < n; ++i) {
            while (!stk.empty() && heights[i] < heights[stk.top()]) {
                int h = heights[stk.top()];
                stk.pop();
                int w = i - stk.top() - 1;
                maxArea = max(maxArea, h * w);
            }
            stk.push(i);
        }
        return maxArea;
    }
};

## J 消防局的设立

#include <iostream>
#include <vector>
#include <algorithm>

using namespace std;

int main() {
    int n;
    cin >> n;
    vector<int> parent(n + 1, 0);
    vector<vector<int>> adj(n + 1);
    for (int i = 2; i <= n; ++i) {
        int p;
        cin >> p;
        parent[i] = p;
        adj[i].push_back(p);
        adj[p].push_back(i);
    }
    parent[1] = 0;

    vector<int> depth(n + 1, 0);
    for (int i = 2; i <= n; ++i) {
        depth[i] = depth[parent[i]] + 1;
    }

    vector<int> nodes(n);
    for (int i = 0; i < n; ++i) nodes[i] = i + 1;
    sort(nodes.begin(), nodes.end(),
         [&](int a, int b) { return depth[a] > depth[b]; });

    vector<bool> covered(n + 1, false);
    int ans = 0;

    for (int u : nodes) {
        if (covered[u]) continue;

        int v;
        if (parent[u] != 0 && parent[parent[u]] != 0) {
            v = parent[parent[u]];
        } else if (parent[u] != 0) {
            v = parent[u];
        } else {
            v = u;
        }

        ++ans;
        covered[v] = true;
        for (int w : adj[v]) {
            covered[w] = true;
            for (int x : adj[w]) {
                covered[x] = true;
            }
        }
    }

    cout << ans << endl;
    return 0;
}